In [1]:
import pydicom

/Users/rumethsandinu/Documents/IRP/NeoBreath/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# view dcm properties
IMAGE_PATH = "../Datasets/CPTAC-LUAD/manifest-1754069116660/CPTAC-LUAD/C3L-00263/12-05-1999-NA-CT CHEST WO CONTRAST-68461/1.000000-SCOUT-19241/1-1.dcm"

properties = pydicom.dcmread(IMAGE_PATH)
print(properties)

Dataset.file_meta -------------------------------
(0002, 0000) File Meta Information Group Length  UL: 206
(0002, 0001) File Meta Information Version       OB: b'\x00\x01'
(0002, 0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002, 0003) Media Storage SOP Instance UID      UI: 1.3.6.1.4.1.14519.5.2.1.2857.5885.315022983099148849026967763329
(0002, 0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002, 0012) Implementation Class UID            UI: 1.3.6.1.4.1.22213.1.143
(0002, 0013) Implementation Version Name         SH: '0.5'
(0002, 0016) Source Application Entity Title     AE: 'POSDA'
-------------------------------------------------
(0008, 0005) Specific Character Set              CS: 'ISO_IR 100'
(0008, 0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'LOCALIZER']
(0008, 0012) Instance Creation Date              DA: '19991205'
(0008, 0013) Instance Creation Time              TM: '115626'
(0008, 0016) SOP Class UID      

In [3]:
import os
import pydicom
from collections import defaultdict
from tqdm import tqdm


In [4]:
# dataset path
DATASET_PATH = "../Datasets/CPTAC-LUAD/manifest-1754069116660/CPTAC-LUAD"

sop_counts = defaultdict(int)

# collect all DICOM files
all_dcm_files = [os.path.join(root, f)
                 for root, _, files in os.walk(DATASET_PATH)
                 for f in files if f.endswith(".dcm")]

for filepath in tqdm(all_dcm_files, desc="Scanning DICOM files"):

    try:
        dcm = pydicom.dcmread(filepath, stop_before_pixels=True)
        sop_uid = getattr(dcm.file_meta, "MediaStorageSOPClassUID", None)
        if sop_uid:
            sop_counts[sop_uid.name] += 1  # human-readable name
    except Exception as e:
        print(f"Error reading {filepath}: {e}")

#  summary
print("\n===== SOP Class UID Distribution =====\n")
for sop_name, count in sop_counts.items():
    print(f"{sop_name}: {count} files")

Scanning DICOM files: 100%|██████████| 63603/63603 [00:59<00:00, 1072.08it/s]


===== SOP Class UID Distribution =====

CT Image Storage: 56295 files
MR Image Storage: 758 files
Positron Emission Tomography Image Storage: 6020 files
Secondary Capture Image Storage: 530 files


In [5]:
img_type_counts = defaultdict(int)

# collect all DICOM files
all_dcm_files = [os.path.join(root, f)
                 for root, _, files in os.walk(DATASET_PATH)
                 for f in files if f.endswith(".dcm")]

for filepath in tqdm(all_dcm_files, desc="Scanning DICOM files"):
    try:
        dcm = pydicom.dcmread(filepath, stop_before_pixels=True)

        # extract Image Type (0008,0008)
        img_type = getattr(dcm, "ImageType", None)
        if img_type:

            # convert to tuple so it's hashable
            img_type_counts[tuple(img_type)] += 1
    except Exception as e:
        print(f"Error reading {filepath}: {e}")

# summary
print("\n===== Image Type Distribution =====\n")
for img_type, count in sorted(img_type_counts.items(), key=lambda x: -x[1]):
    print(f"{list(img_type)}: {count} files")

Scanning DICOM files: 100%|██████████| 63603/63603 [00:59<00:00, 1073.41it/s]


===== Image Type Distribution =====

['ORIGINAL', 'PRIMARY', 'AXIAL']: 14303 files
['ORIGINAL', 'PRIMARY', 'AXIAL', 'HELIX']: 11226 files
['ORIGINAL', 'PRIMARY', 'AXIAL', 'CT_SOM5 SPI']: 8521 files
['ORIGINAL', 'PRIMARY']: 6020 files
['DERIVED', 'SECONDARY', 'REFORMATTED', 'AVERAGE']: 5647 files
['DERIVED', 'PRIMARY', 'AXIAL', 'CT_SOM5 MPR']: 2511 files
['DERIVED', 'SECONDARY', 'REFORMATTED', 'MIP']: 1778 files
['ORIGINAL', 'PRIMARY', 'AXIAL', 'VOLUME']: 1631 files
['DERIVED', 'SECONDARY', 'OTHER', 'CSA MPR', '', 'CSAPARALLEL', 'AXIAL', 'CT_SOM5 SPI']: 1301 files
['DERIVED', 'SECONDARY', 'MPR']: 1146 files
['ORIGINAL', 'PRIMARY', 'AXIAL', 'GSI MONO']: 772 files
['DERIVED', 'SECONDARY', 'AXIAL', 'CT_SOM5 SPI']: 703 files
['DERIVED', 'SECONDARY', 'MPR', 'MIP']: 596 files
['DERIVED', 'PRIMARY', 'AXIAL', 'CT_SOM5 SPO']: 592 files
['ORIGINAL', 'PRIMARY', 'OTHER']: 585 files
['DERIVED', 'SECONDARY', 'AXIAL']: 578 files
['DERIVED', 'SECONDARY', 'AXIAL', 'MONO_ENERGY']: 469 files
['DERIVED', 

In [6]:
# counters
slice_counts = {"PET": 0, "CT": 0}
patient_counts = {"PET": set(), "CT": set()}

# valid SOP classes
SOP_UIDS = {
    "PET": "1.2.840.10008.5.1.4.1.1.128",
    "CT": "1.2.840.10008.5.1.4.1.1.2"
}

def is_valid_image(dcm):
    """Return True if DICOM is ORIGINAL + PRIMARY and not derived/reformatted"""
    if not hasattr(dcm, "ImageType"):
        return False
    
    img_type = [s.upper() for s in dcm.ImageType]
    if "ORIGINAL" not in img_type or "PRIMARY" not in img_type:
        return False
    
    bad_keywords = ["DERIVED", "SECONDARY", "LOCALIZER", "QC", "MONO", "MPR"]
    if any(bad in img_type for bad in bad_keywords):
        return False
    
    return True

# walk through dataset
for root, _, files in os.walk(DATASET_PATH):
    for f in files:
        if not f.endswith(".dcm"):
            continue
        
        filepath = os.path.join(root, f)
        try:
            dcm = pydicom.dcmread(filepath, stop_before_pixels=True)
            modality = getattr(dcm, "Modality", "")
            sop_uid = str(getattr(dcm, "SOPClassUID", ""))
            patient_id = str(dcm.PatientID)

            if modality == "PT" and sop_uid == SOP_UIDS["PET"] and is_valid_image(dcm):
                slice_counts["PET"] += 1
                patient_counts["PET"].add(patient_id)

            elif modality == "CT" and sop_uid == SOP_UIDS["CT"] and is_valid_image(dcm):
                slice_counts["CT"] += 1
                patient_counts["CT"].add(patient_id)

        except Exception as e:
            print(f"Error reading {filepath}: {e}")

# summary
print(f"PET: {slice_counts['PET']} slices from {len(patient_counts['PET'])} patients")
print(f"CT: {slice_counts['CT']} slices from {len(patient_counts['CT'])} patients")


PET: 6020 slices from 11 patients
CT: 37083 slices from 34 patients


In [7]:
import shutil

In [8]:
OUTPUT_PATH = "../raw/PET/A"

# counters
slice_count = 0
patient_set = set()

# ensure output folder exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# collect all DICOM files
all_dcm_files = [
    os.path.join(root, f)
    for root, _, files in os.walk(DATASET_PATH)
    for f in files if f.endswith(".dcm")
]

# process files
for filepath in tqdm(all_dcm_files, desc="Processing PET DICOM files"):
    try:
        dcm = pydicom.dcmread(filepath, stop_before_pixels=True)
        modality = getattr(dcm, "Modality", "")
        sop_uid = str(getattr(dcm, "SOPClassUID", ""))

        if modality == "PT" and sop_uid == SOP_UIDS["PET"] and is_valid_image(dcm):
            patient_id = str(dcm.PatientID)
            full_patient_id = f"CPTACT-LUAD-{patient_id}"

            dest_dir = os.path.join(OUTPUT_PATH, full_patient_id)
            os.makedirs(dest_dir, exist_ok=True)

            dest_file = os.path.join(dest_dir, os.path.basename(filepath))
            if not os.path.exists(dest_file):
                shutil.copy2(filepath, dest_file)

            slice_count += 1
            patient_set.add(full_patient_id)

    except Exception as e:
        print(f"Error reading {filepath}: {e}")

# summary
print(f"PET: {slice_count} slices copied from {len(patient_set)} patients")

Processing PET DICOM files: 100%|██████████| 63603/63603 [01:01<00:00, 1030.66it/s]

PET: 6020 slices copied from 11 patients


In [9]:
OUTPUT_PATH = "../raw/CT/A"

# counters
slice_count = 0
patient_set = set()

# ensure output folder exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# collect all DICOM files
all_dcm_files = [
    os.path.join(root, f)
    for root, _, files in os.walk(DATASET_PATH)
    for f in files if f.endswith(".dcm")
]

# process files
for filepath in tqdm(all_dcm_files, desc="Processing CT DICOM files"):
    try:
        dcm = pydicom.dcmread(filepath, stop_before_pixels=True)
        modality = getattr(dcm, "Modality", "")
        sop_uid = str(getattr(dcm, "SOPClassUID", ""))

        if modality == "CT" and sop_uid == SOP_UIDS["CT"] and is_valid_image(dcm):
            patient_id = str(dcm.PatientID)
            full_patient_id = f"CPTACT-LUAD-{patient_id}"

            dest_dir = os.path.join(OUTPUT_PATH, full_patient_id)
            os.makedirs(dest_dir, exist_ok=True)

            dest_file = os.path.join(dest_dir, os.path.basename(filepath))
            if not os.path.exists(dest_file):
                shutil.copy2(filepath, dest_file)

            slice_count += 1
            patient_set.add(full_patient_id)

    except Exception as e:
        print(f"Error reading {filepath}: {e}")

# summary
print(f"CT: {slice_count} slices copied from {len(patient_set)} patients")

Processing CT DICOM files: 100%|██████████| 63603/63603 [01:16<00:00, 826.06it/s] 

CT: 37083 slices copied from 34 patients
